In [0]:
dbutils.widgets.dropdown("data_source", "products", ["products", "categories"], "Data Source")
dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

print(f"Selected source: {data_source} and catalog: {catalog}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType, TimestampType, DoubleType

from pyspark.sql.window import Window

from delta.tables import DeltaTable

In [0]:
silver_products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("gender_target", StringType(), True),
    StructField("size_uk", DoubleType(), True),
    StructField("colour", StringType(), True),
    StructField("material", StringType(), True),
    StructField("cost_price", DoubleType(), True),
    StructField("retail_price", DoubleType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("launch_date", DateType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
])

silver_categories_schema = StructType([
    StructField("category_id", StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("parent_category_id", StringType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True),
])

DATA_SOURCE_CONFIG = {
    "products": {
            "pk": "product_id",
            "schema": silver_products_schema

    },
    "categories": {
            "pk": "category_id",
            "schema": silver_categories_schema
    }
}

config= DATA_SOURCE_CONFIG[data_source]

source_table = f"{catalog}.os_stepright.bronze_{data_source}_valid"

target_table = f"{catalog}.os_stepright.silver_{data_source}_scd1"

checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/silver_{data_source}_scd1"

pk = config['pk']

schema= config['schema']

In [0]:
print(pk)
print(schema)

In [0]:
print(f"source_table is: [{source_table}] & target table is: [{target_table}]")
print(f"checkpoint is : {checkpoint_loc}")

In [0]:
def target_table_existense_check(target_table, schema):

    if not spark.catalog.tableExists(target_table):

        empty_df = spark.createDataFrame([], schema)
        empty_df.write.format("delta").saveAsTable(target_table)

    else:
        pass



In [0]:
def read_source_table(source_table):
    source_df = (spark.readStream.table(source_table)
    )

    if data_source=='products':
        source_df= source_df.withColumn("launch_date", F.col("launch_date").cast("date"))

    return source_df

In [0]:
target_table_existense_check(target_table, schema)

In [0]:
def process_batch(batch_df, batch_id):

    window_spec = Window.partitionBy(F.col(pk)).orderBy(F.col("_ingested_at").desc())

    dedup_df = (batch_df
        .withColumn("rn", F.row_number().over(window_spec))
        .where("rn=1")
        .drop("rn")
    )

    target_delta = DeltaTable.forName(spark, target_table)

    (target_delta.alias("t")
        .merge(dedup_df.alias("s"), f"t.{pk} = s.{pk}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

source_stream_df = read_source_table(source_table)

(source_stream_df.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", checkpoint_loc)
    .trigger(availableNow=True)
    .start()
)